## Import Region data into MongoDB
The goal is to import the following location (region) data into our database:

- Countries and Continents (with geometry and coordinates)
- States and Districts (with geometry and coordinates, postal code, address region)
- Cities, localities and neighbourhoods (with geometry, postal code, address region, address locality, street address)

Having this data locally makes it cheaper and reduces latency for us.

#### Install Dependencies

In [2]:
!uv sync --python 3.13

zsh:1: command not found: uv


#### Import Libraries

In [3]:
import geopandas as gpd
import pandas as pd
import os
import httpx
import zipfile
import tenacity
import pathlib
from concurrent.futures import ProcessPoolExecutor

### Data Extraction Helpers

Let's define helper functions to help us download and cache datasets for processing.

In [4]:

DATA_DIR = "data"

@tenacity.retry(
    stop=tenacity.stop_after_attempt(5),            # retry up to 5 times
    wait=tenacity.wait_exponential(multiplier=1, min=2, max=20),  # exponential backoff
    reraise=True
)
async def fetch_file(url: str, filepath: str):
    """Fetch a file asynchronously with retries."""
    timeout = httpx.Timeout(30.0, connect=10.0)
    async with httpx.AsyncClient(timeout=timeout) as client:
        response = await client.get(url)
        response.raise_for_status()
        with open(filepath, "wb") as f:
            f.write(response.content)
    return filepath


async def download_and_extract(url: str, extract_to: str = DATA_DIR, force: bool = False) -> list[str]:
    """
    Async download and extract a file (zip or direct) from a URL.

    Retries handled by tenacity.
    """
    os.makedirs(extract_to, exist_ok=True)
    filename = os.path.basename(url)
    filepath = os.path.join(extract_to, filename)

    if not os.path.exists(filepath) or force:
        print(f"Downloading from {url}...")
        await fetch_file(url, filepath)
        print("Download complete.")
    else:
        print(f"File already exists: {filepath}, skipping download.")

    extracted_files = []
    if zipfile.is_zipfile(filepath):
        with zipfile.ZipFile(filepath, "r") as z:
            z.extractall(extract_to)
            extracted_files = [os.path.join(extract_to, name) for name in z.namelist()]
        print("Extraction complete.")
    else:
        extracted_files = [filepath]

    return extracted_files

### Extract Continent Data

Let's start from the top- extract continent data first

In [5]:
def build_continent_aliases(world: gpd.GeoDataFrame):
    """Build a flat list of alternative names for each continent."""

    continent_altnames = {}
    for continent, group in world.groupby("CONTINENT"):
        continent_altnames[continent] = []
    return continent_altnames


async def load_continents():
    """Load continent geometries with name, centroid coordinates, and alternative names."""
    extracted_files = await download_and_extract(
        "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
    )

    shapefile = next((f for f in extracted_files if f.endswith(".shp")), None)
    if shapefile is None:
        raise FileNotFoundError("No shapefile found in extracted dataset.")

    # Load countries
    world = gpd.read_file(shapefile)
    
    print("Continent data sample:")
    display(world.head())

    # Dissolve into continents
    continents = world.dissolve(by="CONTINENT").reset_index()[["CONTINENT", "geometry"]]

    # Reproject to a projected CRS before centroid calculation
    continents_proj = continents.to_crs("ESRI:54034")  # Equal Earth projection
    centroids_proj = continents_proj.centroid

    # Convert centroids back to lat/lon (WGS84)
    centroids_wgs84 = gpd.GeoSeries(centroids_proj, crs="ESRI:54034").to_crs("EPSG:4326")

    # Assign coordinates as [lat, lon]
    continents["coordinates"] = centroids_wgs84.apply(lambda p: [p.y, p.x])

    # Build alt names
    continent_altnames = build_continent_aliases(world)
    continents["aliases"] = continents["CONTINENT"].map(continent_altnames)
    
    continents["level"] = "continent"

    # Rename column for clarity
    continents = continents.rename(columns={"CONTINENT": "name"})

    return continents

continents = await load_continents()
continents

File already exists: data/ne_110m_admin_0_countries.zip, skipping download.
Extraction complete.
Continent data sample:


,featurecla,scalerank,LABELRANK,SOVEREIGNT,SOV_A3,ADM0_DIF,LEVEL,TYPE,TLC,ADMIN,...,FCLASS_TR,FCLASS_ID,FCLASS_PL,FCLASS_GR,FCLASS_IT,FCLASS_NL,FCLASS_SE,FCLASS_BD,FCLASS_UA,geometry
0,Admin-0 country,1,6,Fiji,FJI,0,2,Sovereign country,1,Fiji,...,None,None,None,None,None,None,None,None,None,"MULTIPOLYGON (((180 -16.06713, 180 -16.55522, ..."
1,Admin-0 country,1,3,United Republic of Tanzania,TZA,0,2,Sovereign country,1,United Republic of Tanzania,...,None,None,None,None,None,None,None,None,None,"POLYGON ((33.90371 -0.95, 34.07262 -1.05982, 3..."
2,Admin-0 country,1,7,Western Sahara,SAH,0,2,Indeterminate,1,Western Sahara,...,Unrecognized,Unrecognized,Unrecognized,None,None,Unrecognized,None,None,None,"POLYGON ((-8.66559 27.65643, -8.66512 27.58948..."
3,Admin-0 country,1,2,Canada,CAN,0,2,Sovereign country,1,Canada,...,None,None,None,None,None,None,None,None,None,"MULTIPOLYGON (((-122.84 49, -122.97421 49.0025..."
4,Admin-0 country,1,2,United States of America,US1,1,2,Country,1,United States of America,...,None,None,None,None,None,None,None,None,None,"MULTIPOLYGON (((-122.84 49, -120 49, -117.0312..."


,name,geometry,coordinates,aliases,level
0,Africa,"MULTIPOLYGON (((-11.43878 6.78592, -11.70819 6...","[6.032181157557932, 18.40040406024284]",[],continent
1,Antarctica,"MULTIPOLYGON (((-61.13898 -79.98137, -60.61012...","[-75.50522868903988, 35.64346078762372]",[],continent
2,Asia,"MULTIPOLYGON (((48.67923 14.0032, 48.23895 13....","[29.888226373617375, 84.71223389814556]",[],continent
3,Europe,"MULTIPOLYGON (((-53.55484 2.3349, -53.77852 2....","[56.65066091381333, 73.8348995832024]",[],continent
4,North America,"MULTIPOLYGON (((-155.22217 19.23972, -155.5421...","[47.14289012790487, -96.36461725674786]",[],continent
5,Oceania,"MULTIPOLYGON (((147.91405 -43.21152, 147.56456...","[-24.573805155566546, 136.50696195370307]",[],continent
6,Seven seas (open ocean),"POLYGON ((68.935 -48.625, 69.58 -48.94, 70.525...","[-49.30442568487861, 69.53123972063625]",[],continent
7,South America,"MULTIPOLYGON (((-68.63999 -55.58002, -69.2321 ...","[-13.587586113083029, -60.64773056426989]",[],continent


In [6]:
# Drop Seven Seas
# note: Oceania is taken as a continent, and not Australia here- which is referred to as a country.
continents = continents[continents["name"] != "Seven seas (open ocean)"]
continents = continents.reset_index(drop=True)
continents

,name,geometry,coordinates,aliases,level
0,Africa,"MULTIPOLYGON (((-11.43878 6.78592, -11.70819 6...","[6.032181157557932, 18.40040406024284]",[],continent
1,Antarctica,"MULTIPOLYGON (((-61.13898 -79.98137, -60.61012...","[-75.50522868903988, 35.64346078762372]",[],continent
2,Asia,"MULTIPOLYGON (((48.67923 14.0032, 48.23895 13....","[29.888226373617375, 84.71223389814556]",[],continent
3,Europe,"MULTIPOLYGON (((-53.55484 2.3349, -53.77852 2....","[56.65066091381333, 73.8348995832024]",[],continent
4,North America,"MULTIPOLYGON (((-155.22217 19.23972, -155.5421...","[47.14289012790487, -96.36461725674786]",[],continent
5,Oceania,"MULTIPOLYGON (((147.91405 -43.21152, 147.56456...","[-24.573805155566546, 136.50696195370307]",[],continent
6,South America,"MULTIPOLYGON (((-68.63999 -55.58002, -69.2321 ...","[-13.587586113083029, -60.64773056426989]",[],continent


### Extract Country Data
Next, let's import all the countries in the continents we have extracted so far.

In [7]:
def build_country_aliases(countries: gpd.GeoDataFrame):
    """Build a flat list of alternative names for each country."""
    alt_name_columns = [
        col for col in countries.columns
        if col.startswith("NAME_") or col in ["NAME_ALT", "NAME_LONG", "FORMAL_EN", "FORMAL_FR"]
    ]

    country_altnames = {}
    for _, row in countries.iterrows():
        names = set()
        for col in alt_name_columns:
            v = row[col]
            if v not in [None, "", -99] and isinstance(v, str):
                names.add(str(v))  # ensure all names are strings
        country_altnames[row["ADMIN"]] = sorted(names)

    return country_altnames


async def load_countries():
    """Load all countries with name, centroid coordinates, and alternative names."""
    extracted_files = await download_and_extract(
        "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
    )

    shapefile = next((f for f in extracted_files if f.endswith(".shp")), None)
    if shapefile is None:
        raise FileNotFoundError("No shapefile found in extracted dataset.")

    # Load countries
    countries = gpd.read_file(shapefile)
    
    print("Country data sample:")
    display(countries.head())

    # Build aliases before dropping columns
    country_altnames = build_country_aliases(countries)

    # Keep only useful columns
    countries = countries[['ADMIN', 'ISO_A2', 'geometry']]

    # Project to a suitable projected CRS for accurate centroid computation
    countries_proj = countries.to_crs("ESRI:54034")  # Equal Earth
    centroids_proj = countries_proj.centroid
    centroids_wgs84 = gpd.GeoSeries(centroids_proj, crs="ESRI:54034").to_crs("EPSG:4326")

    # Assign coordinates as [lat, lon]
    countries['coordinates'] = centroids_wgs84.apply(lambda p: [p.y, p.x])

    # Map aliases
    countries['aliases'] = countries['ADMIN'].map(country_altnames)

    # Optionally rename columns for clarity
    countries = countries.rename(columns={"ADMIN": "name", "ISO_A2": "iso_a2"})
    
    countries["level"] = "country"

    return countries

countries = await load_countries()
countries

File already exists: data/ne_110m_admin_0_countries.zip, skipping download.
Extraction complete.
Country data sample:


,featurecla,scalerank,LABELRANK,SOVEREIGNT,SOV_A3,ADM0_DIF,LEVEL,TYPE,TLC,ADMIN,...,FCLASS_TR,FCLASS_ID,FCLASS_PL,FCLASS_GR,FCLASS_IT,FCLASS_NL,FCLASS_SE,FCLASS_BD,FCLASS_UA,geometry
0,Admin-0 country,1,6,Fiji,FJI,0,2,Sovereign country,1,Fiji,...,None,None,None,None,None,None,None,None,None,"MULTIPOLYGON (((180 -16.06713, 180 -16.55522, ..."
1,Admin-0 country,1,3,United Republic of Tanzania,TZA,0,2,Sovereign country,1,United Republic of Tanzania,...,None,None,None,None,None,None,None,None,None,"POLYGON ((33.90371 -0.95, 34.07262 -1.05982, 3..."
2,Admin-0 country,1,7,Western Sahara,SAH,0,2,Indeterminate,1,Western Sahara,...,Unrecognized,Unrecognized,Unrecognized,None,None,Unrecognized,None,None,None,"POLYGON ((-8.66559 27.65643, -8.66512 27.58948..."
3,Admin-0 country,1,2,Canada,CAN,0,2,Sovereign country,1,Canada,...,None,None,None,None,None,None,None,None,None,"MULTIPOLYGON (((-122.84 49, -122.97421 49.0025..."
4,Admin-0 country,1,2,United States of America,US1,1,2,Country,1,United States of America,...,None,None,None,None,None,None,None,None,None,"MULTIPOLYGON (((-122.84 49, -120 49, -117.0312..."


,name,iso_a2,geometry,coordinates,aliases,level
0,Fiji,FJ,"MULTIPOLYGON (((180 -16.06713, 180 -16.55522, ...","[-17.31277520899394, 163.77640725005324]","[Fidji, Fidschi, Fidzsi-szigetek, Fidżi, Figi,...",country
1,United Republic of Tanzania,TZ,"POLYGON ((33.90371 -0.95, 34.07262 -1.05982, 3...","[-6.237730725476361, 34.74766857623911]","[Tansania, Tanzania, Tanzanie, Tanzanya, Tanzá...",country
2,Western Sahara,EH,"POLYGON ((-8.66559 27.65643, -8.66512 27.58948...","[24.24928060599045, -12.162183576752351]","[Batı Sahra, Nyugat-Szahara, Sahara Barat, Sah...",country
3,Canada,CA,"MULTIPOLYGON (((-122.84 49, -122.97421 49.0025...","[57.62966510888876, -97.64486003045715]","[Canada, Canadá, Kanada, Καναδάς, Канада, קנדה...",country
4,United States of America,US,"MULTIPOLYGON (((-122.84 49, -120 49, -117.0312...","[42.075007554015286, -107.5447286816333]","[Amerika Birleşik Devletleri, Amerika Serikat,...",country
...,...,...,...,...,...,...
172,Republic of Serbia,RS,"POLYGON ((18.82982 45.90887, 18.82984 45.90888...","[44.21138805382481, 20.828267375143835]","[Republic of Serbia, Serbia, Serbie, Serbien, ...",country
173,Montenegro,ME,"POLYGON ((20.0707 42.58863, 19.80161 42.50009,...","[42.78633660560693, 19.285947508559723]","[Czarnogóra, Karadağ, Montenegro, Montenegró, ...",country
174,Kosovo,-99,"POLYGON ((20.59025 41.85541, 20.52295 42.21787...","[42.57742521451801, 20.895365849100475]","[Kosova, Kosovo, Kosowo, Koszovó, Republic of ...",country
175,Trinidad and Tobago,TT,"POLYGON ((-61.68 10.76, -61.105 10.89, -60.895...","[10.427976277378917, -61.330418679071556]","[Republic of Trinidad and Tobago, Trinidad and...",country


Let's filter out countries with an invalid alpha2 code

In [8]:
countries = countries[countries['iso_a2'] != "-99"]
countries = countries.reset_index(drop=True)
countries


,name,iso_a2,geometry,coordinates,aliases,level
0,Fiji,FJ,"MULTIPOLYGON (((180 -16.06713, 180 -16.55522, ...","[-17.31277520899394, 163.77640725005324]","[Fidji, Fidschi, Fidzsi-szigetek, Fidżi, Figi,...",country
1,United Republic of Tanzania,TZ,"POLYGON ((33.90371 -0.95, 34.07262 -1.05982, 3...","[-6.237730725476361, 34.74766857623911]","[Tansania, Tanzania, Tanzanie, Tanzanya, Tanzá...",country
2,Western Sahara,EH,"POLYGON ((-8.66559 27.65643, -8.66512 27.58948...","[24.24928060599045, -12.162183576752351]","[Batı Sahra, Nyugat-Szahara, Sahara Barat, Sah...",country
3,Canada,CA,"MULTIPOLYGON (((-122.84 49, -122.97421 49.0025...","[57.62966510888876, -97.64486003045715]","[Canada, Canadá, Kanada, Καναδάς, Канада, קנדה...",country
4,United States of America,US,"MULTIPOLYGON (((-122.84 49, -120 49, -117.0312...","[42.075007554015286, -107.5447286816333]","[Amerika Birleşik Devletleri, Amerika Serikat,...",country
...,...,...,...,...,...,...
167,North Macedonia,MK,"POLYGON ((22.38053 42.32026, 22.88137 41.9993,...","[41.60305470453288, 21.697076764970955]","[Bắc Macedonia, Kuzey Makedonya, Macedonia Pół...",country
168,Republic of Serbia,RS,"POLYGON ((18.82982 45.90887, 18.82984 45.90888...","[44.21138805382481, 20.828267375143835]","[Republic of Serbia, Serbia, Serbie, Serbien, ...",country
169,Montenegro,ME,"POLYGON ((20.0707 42.58863, 19.80161 42.50009,...","[42.78633660560693, 19.285947508559723]","[Czarnogóra, Karadağ, Montenegro, Montenegró, ...",country
170,Trinidad and Tobago,TT,"POLYGON ((-61.68 10.76, -61.105 10.89, -60.895...","[10.427976277378917, -61.330418679071556]","[Republic of Trinidad and Tobago, Trinidad and...",country


Let's inspect some sample data

In [9]:
countries[countries["name"] == "India"]

,name,iso_a2,geometry,coordinates,aliases,level
96,India,IN,"POLYGON ((97.32711 28.26158, 97.40256 27.88254...","[22.609142131655616, 79.57356544184115]","[Hindistan, Inde, India, Indie, Indien, Republ...",country


### Extract State Data
Next, let's import all the states in the countries we have extracted so far.

In [10]:
def build_state_aliases(states: gpd.GeoDataFrame):
    """Build flat list of alternative names for each state/province."""
    alt_name_columns = [
        col for col in states.columns
        if col.lower().startswith("name") or col in ["name_alt", "name_local", "name_ascii"]
    ]

    state_altnames = {}
    for _, row in states.iterrows():
        names = set()
        for col in alt_name_columns:
            v = row[col]
            if v not in [None, "", -99] and isinstance(v, str):
                names.add(v)
        state_altnames[row["name"]] = sorted(names)
    return state_altnames


async def load_states():
    """Load all states/provinces worldwide with name, centroid coordinates, and aliases."""
    url = "http://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_1_states_provinces.zip"
    extracted_files = await download_and_extract(url)

    shapefile = next((f for f in extracted_files if f.endswith(".shp")), None)
    if shapefile is None:
        raise FileNotFoundError("No shapefile found in extracted dataset.")

    states = gpd.read_file(shapefile)

    print("State data sample:")
    display(states.head())

    # Build aliases before dropping columns
    state_altnames = build_state_aliases(states)

    # Keep useful columns
    states = states[['name', "iso_a2",  'admin', 'geometry']]

   # Clean Alpha2 codes
    states['iso_a2'] = states['iso_a2'].apply(lambda x: x if str(x) != "-99" else None)

    # Project for accurate centroid
    states_proj = states.to_crs("ESRI:54034")  # Equal Earth projection
    centroids_proj = states_proj.centroid
    centroids_wgs84 = gpd.GeoSeries(centroids_proj, crs="ESRI:54034").to_crs("EPSG:4326")
    states['coordinates'] = centroids_wgs84.apply(lambda p: [p.y, p.x])

    # Map aliases
    states['aliases'] = states['name'].map(state_altnames)

    # Rename columns for clarity
    states = states.rename(columns={
        "admin": "country"
    })
    
    states["level"] = "state"

    return states

states = await load_states()
states

File already exists: data/ne_10m_admin_1_states_provinces.zip, skipping download.
Extraction complete.
State data sample:


,featurecla,scalerank,adm1_code,diss_me,iso_3166_2,wikipedia,iso_a2,adm0_sr,name,name_alt,...,FCLASS_ID,FCLASS_PL,FCLASS_GR,FCLASS_IT,FCLASS_NL,FCLASS_SE,FCLASS_BD,FCLASS_UA,FCLASS_TLC,geometry
0,Admin-1 states provinces,3,ARG-1309,1309,AR-E,None,AR,1,Entre Ríos,Entre-Rios,...,None,None,None,None,None,None,None,None,None,"POLYGON ((-58.20011 -32.44713, -58.20012 -32.4..."
1,Admin-1 states provinces,6,URY-8,8,UY-PA,None,UY,1,Paysandú,None,...,None,None,None,None,None,None,None,None,None,"POLYGON ((-58.20012 -32.4472, -58.20011 -32.44..."
2,Admin-1 states provinces,2,IDN-1185,1185,ID-KI,None,ID,5,Kalimantan Timur,Kaltim,...,None,None,None,None,None,None,None,None,None,"MULTIPOLYGON (((117.70361 4.16341, 117.70361 4..."
3,Admin-1 states provinces,6,MYS-1186,1186,MY-12,None,MY,5,Sabah,North Borneo,...,None,None,None,None,None,None,None,None,None,"MULTIPOLYGON (((117.70361 4.16341, 117.70361 4..."
4,Admin-1 states provinces,3,CHL-2694,2694,CL-AP,None,CL,1,Arica y Parinacota,None,...,None,None,None,None,None,None,None,None,None,"POLYGON ((-69.51009 -17.50659, -69.50611 -17.5..."


,name,iso_a2,country,geometry,coordinates,aliases,level
0,Entre Ríos,AR,Argentina,"POLYGON ((-58.20011 -32.44713, -58.20012 -32.4...","[-32.02547013695879, -59.206689862401774]","[Entre Ríos, Entre Ríos eyaleti, Entre-Rios, Έ...",state
1,Paysandú,UY,Uruguay,"POLYGON ((-58.20012 -32.4472, -58.20011 -32.44...","[-32.082363780360275, -57.345107641978004]","[Departemen Paysandú, Paysandu Departmanı, Pay...",state
2,Kalimantan Timur,ID,Indonesia,"MULTIPOLYGON (((117.70361 4.16341, 117.70361 4...","[1.352699778087046, 116.38394709926088]","[Borneo Wschodnie, Doğu Kalimantan, East Kalim...",state
3,Sabah,MY,Malaysia,"MULTIPOLYGON (((117.70361 4.16341, 117.70361 4...","[5.345010895470702, 117.13171580564357]","[North Borneo, Sabah, Sabá, Σαμπάχ, Сабах, סבה...",state
4,Arica y Parinacota,CL,Chile,"POLYGON ((-69.51009 -17.50659, -69.50611 -17.5...","[-18.485850925370954, -69.63872074583377]","[Arica e Parinacota, Arica et Parinacota, Aric...",state
...,...,...,...,...,...,...,...
4591,Isla,MT,Malta,"POLYGON ((14.51073 35.88896, 14.51537 35.89113...","[35.88612913557184, 14.514443273773145]","[Isla, Isla|Città Invicta, L-Isla, Senglea, Σε...",state
4592,Cospicua,MT,Malta,"POLYGON ((14.52937 35.88156, 14.527 35.87772, ...","[35.87927143250959, 14.52043248348041]","[Bormla, Cospicua, Κοσπίκουα, Бормла, קוספיקאה...",state
4593,Pietà,MT,Malta,"POLYGON ((14.49801 35.89251, 14.49616 35.8883,...","[35.88921922041928, 14.492441073423253]","[Pietà, Πιετά, Пьета, Пієта, פיאטה, بييتا, پیئ...",state
4594,Msida,MT,Malta,"POLYGON ((14.48663 35.88636, 14.48231 35.88979...","[35.894557532084654, 14.48254329099217]","[Msida, Μσίντα, Мсида, Мсіда, מסידה, مسيدا, مس...",state


Let's inspect some sample state data

In [11]:
states[states["name"] == "Tamil Nadu"]

,name,iso_a2,country,geometry,coordinates,aliases,level
2242,Tamil Nadu,IN,India,"MULTIPOLYGON (((79.84734 10.80964, 79.84986 10...","[11.007117336162471, 78.40069620443494]","[Tamil Nadu, Tamilnadu, Tamilnádu, Ταμίλ Ναντο...",state


### Extract City Data
Next, let's import all the cities in the states we have extracted so far.

In [24]:
CHUNK_SIZE = 100_000  # rows per chunk
PARQUET_DIR = pathlib.Path(f"{DATA_DIR}/geonames_parquet")
PARQUET_DIR.mkdir(exist_ok=True, parents=True)

def build_aliases(row, alt_col: str = "NAMEALT", name_col: str = "NAME"):
    names = set()
    for col in [name_col, alt_col]:
        v = row.get(col)
        if v and isinstance(v, str):
            # Split by comma and strip whitespace
            for name in v.split(","):
                name = name.strip()
                if name:
                    names.add(name)
    return sorted(names)


def process_chunk(chunk, chunk_index):
    """Process a single chunk and write to Parquet."""
    chunk['aliases'] = chunk.apply(build_aliases, axis=1)
    chunk['name'] = chunk['NAME']
    chunk['iso_a2'] = chunk['country_code']
    chunk['coordinates'] = chunk.apply(lambda r: [float(r['latitude']), float(r['longitude'])], axis=1)
    chunk['street_address'] = None
    chunk['postal_code'] = None
    chunk["address_region"] = None

    address_chunk = chunk[[
        'name', 'street_address', 'admin1_code', "address_region", 'postal_code', 'iso_a2', 'coordinates', 'aliases'
    ]]

    parquet_path = PARQUET_DIR / f"geonames_chunk_{chunk_index}.parquet"
    address_chunk.to_parquet(parquet_path, index=False)
    print(f"Chunk {chunk_index} saved to {parquet_path}")
    return parquet_path

async def load_cities_localities_chunked_mp():
    geonames_url = "http://download.geonames.org/export/dump/allCountries.zip"
    extracted_files = await download_and_extract(geonames_url)
    geonames_file = next(pathlib.Path(f) for f in extracted_files if pathlib.Path(f).suffix == ".txt")

    columns = [
        "geonameid", "NAME", "NAMEASCII", "NAMEALT", "latitude", "longitude",
        "feature_class", "feature_code", "country_code", "cc2", "admin1_code",
        "admin2_code", "admin3_code", "admin4_code", "population", "elevation",
        "dem", "timezone", "modification_date"
    ]

    parquet_files = []

    # Read CSV in chunks
    chunks = pd.read_csv(
        geonames_file,
        sep="\t",
        header=None,
        names=columns,
        dtype=str,
        chunksize=CHUNK_SIZE
    )
    
    sample_chunks = pd.read_csv(
        geonames_file,
        sep="\t",
        header=None,
        names=columns,
        dtype=str,
        chunksize=CHUNK_SIZE
    )
    
    sample_df = next(sample_chunks)
    
    print("Localities / Cities data sample:")
    display(sample_df.head())  # Display first chunk sample

    with ProcessPoolExecutor() as executor:
        futures = []
        for i, chunk in enumerate(chunks):
            # Optional: filter populated places
            # chunk = chunk[chunk['feature_class'] == 'P']
            futures.append(executor.submit(process_chunk, chunk, i + 1))

        for future in futures:
            parquet_files.append(future.result())

    print(f"All chunks saved. Total chunks: {len(parquet_files)}")
    return parquet_files

parquet_files = await load_cities_localities_chunked_mp()

File already exists: data/allCountries.zip, skipping download.
Extraction complete.
Localities / Cities data sample:


,geonameid,NAME,NAMEASCII,NAMEALT,latitude,longitude,feature_class,feature_code,country_code,cc2,admin1_code,admin2_code,admin3_code,admin4_code,population,elevation,dem,timezone,modification_date
0,2994701,Roc Meler,Roc Meler,"Roc Mele,Roc Meler,Roc Mélé",42.58765,1.7418,T,PK,AD,"AD,FR",02,NaN,NaN,NaN,0,2811,2348,Europe/Andorra,2023-10-03
1,3017832,Pic de les Abelletes,Pic de les Abelletes,"Pic de la Font-Negre,Pic de la Font-Nègre,Pic ...",42.52535,1.73343,T,PK,AD,FR,A9,66,663,66146,0,NaN,2411,Europe/Andorra,2014-11-05
2,3017833,Estany de les Abelletes,Estany de les Abelletes,"Estany de les Abelletes,Etang de Font-Negre,Ét...",42.52915,1.73362,H,LK,AD,FR,A9,NaN,NaN,NaN,0,NaN,2260,Europe/Andorra,2014-11-05
3,3023203,Port Vieux de la Coume d’Ose,Port Vieux de la Coume d'Ose,"Port Vieux de Coume d'Ose,Port Vieux de Coume ...",42.62568,1.61823,T,PASS,AD,NaN,00,NaN,NaN,NaN,0,NaN,2687,Europe/Andorra,2014-11-05
4,3029315,Port de la Cabanette,Port de la Cabanette,"Port de la Cabanette,Porteille de la Cabanette",42.6,1.73333,T,PASS,AD,"AD,FR",B3,09,091,09139,0,NaN,2379,Europe/Andorra,2014-11-05


Chunk 1 saved to data/geonames_parquet/geonames_chunk_1.parquetChunk 2 saved to data/geonames_parquet/geonames_chunk_2.parquet

Chunk 3 saved to data/geonames_parquet/geonames_chunk_3.parquet
Chunk 4 saved to data/geonames_parquet/geonames_chunk_4.parquet
Chunk 5 saved to data/geonames_parquet/geonames_chunk_5.parquet
Chunk 6 saved to data/geonames_parquet/geonames_chunk_6.parquet
Chunk 7 saved to data/geonames_parquet/geonames_chunk_7.parquet
Chunk 8 saved to data/geonames_parquet/geonames_chunk_8.parquet
Chunk 9 saved to data/geonames_parquet/geonames_chunk_9.parquet
Chunk 10 saved to data/geonames_parquet/geonames_chunk_10.parquet
Chunk 11 saved to data/geonames_parquet/geonames_chunk_11.parquet
Chunk 12 saved to data/geonames_parquet/geonames_chunk_12.parquet
Chunk 13 saved to data/geonames_parquet/geonames_chunk_13.parquet
Chunk 14 saved to data/geonames_parquet/geonames_chunk_14.parquet
Chunk 15 saved to data/geonames_parquet/geonames_chunk_15.parquet
Chunk 16 saved to data/geona

In [25]:
# Example of batch processing from Parquet files
def process_parquet_batches(parquet_files):
    """Yield DataFrames in batches from multiple Parquet files."""
    for file in parquet_files:
        yield pd.read_parquet(file)

# Usage
for batch_df in process_parquet_batches(parquet_files):
    # Do something with batch_df
    display(batch_df.head())

,name,street_address,admin1_code,address_region,postal_code,iso_a2,coordinates,aliases
0,Roc Meler,None,02,None,None,AD,"[42.58765, 1.7418]","[Roc Mele, Roc Meler, Roc Mélé]"
1,Pic de les Abelletes,None,A9,None,None,AD,"[42.52535, 1.73343]","[Pic de la Font-Negre, Pic de la Font-Nègre, P..."
2,Estany de les Abelletes,None,A9,None,None,AD,"[42.52915, 1.73362]","[Estany de les Abelletes, Etang de Font-Negre,..."
3,Port Vieux de la Coume d’Ose,None,00,None,None,AD,"[42.62568, 1.61823]","[Port Vieux de Coume d'Ose, Port Vieux de Coum..."
4,Port de la Cabanette,None,B3,None,None,AD,"[42.6, 1.73333]","[Port de la Cabanette, Porteille de la Cabanette]"


,name,street_address,admin1_code,address_region,postal_code,iso_a2,coordinates,aliases
0,"Gavar Mediumwave Transmitter, 1350kHz-Antenna,...",None,None,None,None,AM,"[40.40176, 45.19904]","[1350kHz-Antenna, Gavar Mediumwave Transmitter..."
1,"Gavar Mediumwave Transmitter, 1350kHz-Antenna,...",None,None,None,None,AM,"[40.40057, 45.19894]","[1350kHz-Antenna, Gavar Mediumwave Transmitter..."
2,"Gavar Mediumwave Transmitter, 1350kHz-Antenna,...",None,None,None,None,AM,"[40.40092, 45.19965]","[1350kHz-Antenna, Gavar Mediumwave Transmitter..."
3,"Arinj Mediumwave Transmitter, Mast North",None,None,None,None,AM,"[40.2407, 44.6026]","[Arinj Mediumwave Transmitter, Mast North]"
4,"Arinj Mediumwave Transmitter, Mast South",None,None,None,None,AM,"[40.23978, 44.60263]","[Arinj Mediumwave Transmitter, Mast South]"


,name,street_address,admin1_code,address_region,postal_code,iso_a2,coordinates,aliases
0,Mampangui,None,09,None,None,AO,"[-15.41706, 13.95008]",[Mampangui]
1,Iendja,None,09,None,None,AO,"[-15.45391, 13.9851]",[Iendja]
2,Missão da Tchihita,None,09,None,None,AO,"[-15.40745, 13.96728]",[Missão da Tchihita]
3,Tchiqueia,None,09,None,None,AO,"[-15.47215, 13.97999]",[Tchiqueia]
4,Tchimbandja,None,09,None,None,AO,"[-15.49057, 13.98291]",[Tchimbandja]


,name,street_address,admin1_code,address_region,postal_code,iso_a2,coordinates,aliases
0,Perlhof,None,03,None,None,AT,"[48.08333, 16.25]",[Perlhof]
1,Perlegg,None,06,None,None,AT,"[47.06667, 15.68333]",[Perlegg]
2,Perkonnig,None,02,None,None,AT,"[46.63333, 14.4]","[Perkonik, Perkonnig]"
3,Perjen,None,07,None,None,AT,"[47.14665, 10.56936]",[Perjen]
4,Pergkirchen,None,04,None,None,AT,"[48.2448, 14.67533]",[Pergkirchen]


,name,street_address,admin1_code,address_region,postal_code,iso_a2,coordinates,aliases
0,Alder Park,None,02,None,None,AU,"[-32.9318, 151.7012]",[Alder Park]
1,Albury Racecourse,None,02,None,None,AU,"[-36.06184, 146.95428]",[Albury Racecourse]
2,Albury Rifle Range,None,02,None,None,AU,"[-36.05045, 146.89951]",[Albury Rifle Range]
3,Albury Showground,None,02,None,None,AU,"[-36.06503, 146.93605]",[Albury Showground]
4,Albury Sportsground,None,02,None,None,AU,"[-36.08296, 146.9086]",[Albury Sportsground]


,name,street_address,admin1_code,address_region,postal_code,iso_a2,coordinates,aliases
0,The Pines,None,05,None,None,AU,"[-34.02987, 138.22733]",[The Pines]
1,Pinevale,None,05,None,None,AU,"[-34.41626, 139.66397]",[Pinevale]
2,Pinevilla,None,05,None,None,AU,"[-34.46472, 139.09766]",[Pinevilla]
3,Bowey Road,None,05,None,None,AU,"[-35.188, 138.83385]",[Bowey Road]
4,Chesterville Park,None,05,None,None,AU,"[-35.18744, 138.85111]",[Chesterville Park]


... (124 outputs left out)

,name,street_address,admin1_code,address_region,postal_code,iso_a2,coordinates,aliases
0,Laja Uchachaba,None,01,None,None,VE,"[4.80135, -67.73032]",[Laja Uchachaba]
1,Hato Valle Rosa,None,01,None,None,VE,"[5.57778, -66.28333]",[Hato Valle Rosa]
2,Caño Veneno,None,01,None,None,VE,"[4.90595, -67.7196]",[Caño Veneno]
3,Laguna Zamurera,None,01,None,None,VE,"[4.72572, -67.73594]","[Laguna Zamurera, Laguna Zamurere]"
4,Caño Saisa,None,23,None,None,VE,"[8.97056, -72.2]","[Cano Pabellon, Cano Saisa, Caño Pabellón, Cañ..."


,name,street_address,admin1_code,address_region,postal_code,iso_a2,coordinates,aliases
0,Wādī Aar’daq,None,01,None,None,YE,"[13.65, 46.6]","[Wadi Aar'daq, Wādī Aar’daq]"
1,Wādī Aanfar,None,01,None,None,YE,"[14.13333, 46.68333]","[Wadi Aanfar, Wādī Aanfar]"
2,Wādī al A‘af,None,05,None,None,YE,"[14.16706, 47.13639]","[Wadi al A`af, Wādī al A‘af]"
3,Wādī Aadhamer,None,01,None,None,YE,"[14.25, 46.18333]","[Wadi Aadhamer, Wādī Aadhamer]"
4,Az Zunayfir,None,21,None,None,YE,"[16.38333, 46.28333]","[Az Zunayfir, Zunayfir, alznyfr, الزنيفر]"


,name,street_address,admin1_code,address_region,postal_code,iso_a2,coordinates,aliases
0,Uyshoek,None,03,None,None,ZA,"[-28.26639, 29.35172]",[Uyshoek]
1,Uys Heim,None,03,None,None,ZA,"[-27.5309, 29.43701]",[Uys Heim]
2,Uysedoorn,None,07,None,None,ZA,"[-25.13051, 30.05426]",[Uysedoorn]
3,Uysberg,None,03,None,None,ZA,"[-29.08286, 27.58457]",[Uysberg]
4,Uys,None,08,None,None,ZA,"[-28.03144, 22.8659]",[Uys]


,name,street_address,admin1_code,address_region,postal_code,iso_a2,coordinates,aliases
0,Sambwechi,None,01,None,None,ZM,"[-14.58333, 24.35]",[Sambwechi]
1,Sambwe,None,04,None,None,ZM,"[-12.26667, 29.46667]",[Sambwe]
2,Sambwa,None,03,None,None,ZM,"[-12.41667, 33.33333]",[Sambwa]
3,Sambwa,None,03,None,None,ZM,"[-12.41667, 33.33333]",[Sambwa]
4,Sambututu,None,01,None,None,ZM,"[-17.18333, 25.06667]",[Sambututu]
